# extract.ai Responses API comparison demo

This notebook demonstrates the new `extract.ai` OpenAI path using real OpenAI API calls. Set `OPENAI_API_KEY` in your environment before running it.

It compares:

- New default path: Responses API with Structured Outputs and local Pydantic validation.
- Old compatible path: Chat Completions when explicitly selected with the legacy URL.
- Validation behavior: invalid structured data is caught before it reaches the caller as trusted data.
- Metrics: accuracy against expected values, non-empty rate, agreement rate, and elapsed time.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import pandas as pd

import wrangles.extract as extract

## Shared extraction request

Both approaches receive the same source text and the same requested output schema.

In [ ]:
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

model = "gpt-4o-mini"
data = [
    "wrench 25mm",
    "6m cable",
    "screwdriver 3mm",
]
expected_lengths = ["25mm", "6m", "3mm"]

output_schema = {
    "length": {
        "type": "string",
        "description": "Any lengths found in the data such as cm, m, ft, etc.",
        "examples": "22mm",
    }
}

## New approach: Responses API + Structured Outputs

The default `extract.ai` path now sends a Responses API request with a strict JSON schema in `text.format`. This cell makes real API calls and returns one structured result per input row.

In [ ]:
new_start = time.perf_counter()
new_results = extract.ai(
    data,
    api_key=api_key,
    output=output_schema,
    model=model,
    seed=1,
    timeout=60,
    retries=2,
    threads=1,
)
new_elapsed_seconds = time.perf_counter() - new_start

new_results_df = pd.DataFrame(
    {
        "input": data,
        "expected": expected_lengths,
        "Responses API result": [row.get("length", "") for row in new_results],
    }
)
new_results_df

## Old approach: Chat Completions override

The legacy behavior remains available when the caller explicitly passes the Chat Completions URL. This cell makes real API calls through that compatibility path.

In [ ]:
legacy_start = time.perf_counter()
legacy_results = extract.ai(
    data,
    api_key=api_key,
    output=output_schema,
    model=model,
    seed=1,
    temperature=0.2,
    timeout=60,
    retries=2,
    threads=1,
    url="https://api.openai.com/v1/chat/completions",
)
legacy_elapsed_seconds = time.perf_counter() - legacy_start

legacy_results_df = pd.DataFrame(
    {
        "input": data,
        "expected": expected_lengths,
        "Chat Completions result": [row.get("length", "") for row in legacy_results],
    }
)
legacy_results_df

## Result comparison

In [ ]:
comparison_df = pd.DataFrame(
    {
        "input": data,
        "expected": expected_lengths,
        "Responses API result": [row.get("length", "") for row in new_results],
        "Chat Completions result": [row.get("length", "") for row in legacy_results],
    }
)
comparison_df["Responses API correct"] = (
    comparison_df["Responses API result"] == comparison_df["expected"]
)
comparison_df["Chat Completions correct"] = (
    comparison_df["Chat Completions result"] == comparison_df["expected"]
)
comparison_df["same output"] = (
    comparison_df["Responses API result"] == comparison_df["Chat Completions result"]
)
comparison_df

## Metrics

In [ ]:
def extraction_metrics(label, result_column, correct_column, elapsed_seconds):
    values = comparison_df[result_column].fillna("").astype(str)
    return {
        "approach": label,
        "exact-match accuracy": comparison_df[correct_column].mean(),
        "correct rows": int(comparison_df[correct_column].sum()),
        "total rows": len(comparison_df),
        "non-empty rate": values.ne("").mean(),
        "elapsed seconds": round(elapsed_seconds, 3),
        "seconds per row": round(elapsed_seconds / len(comparison_df), 3),
    }


metrics_df = pd.DataFrame(
    [
        extraction_metrics(
            "Responses API",
            "Responses API result",
            "Responses API correct",
            new_elapsed_seconds,
        ),
        extraction_metrics(
            "Chat Completions",
            "Chat Completions result",
            "Chat Completions correct",
            legacy_elapsed_seconds,
        ),
    ]
)

agreement_rate = comparison_df["same output"].mean()
metrics_df["agreement rate"] = agreement_rate
metrics_df

For this small demo, the strongest signal is exact-match accuracy against known expected outputs. The new approach is better when it preserves or improves accuracy while adding strict schema enforcement and local Pydantic validation. Runtime is useful, but it can vary with network conditions and model load.

## Quick metric charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

metrics_df.set_index("approach")[[
    "exact-match accuracy",
    "non-empty rate",
    "agreement rate",
]].plot(kind="bar", ax=axes[0], ylim=(0, 1), rot=0)
axes[0].set_title("Quality metrics")
axes[0].set_ylabel("Rate")
axes[0].legend(loc="lower right")

metrics_df.set_index("approach")[["seconds per row"]].plot(
    kind="bar",
    ax=axes[1],
    rot=0,
    legend=False,
    color="#4c78a8",
)
axes[1].set_title("Latency")
axes[1].set_ylabel("Seconds per row")

plt.tight_layout()

## Extended benchmark

This section compares the two approaches across different row counts and request settings. The default matrix is intentionally small because every row is a live API call. Increase `benchmark_row_counts` or add `benchmark_configs` when you want a broader evaluation.

In [ ]:
benchmark_cases = [
    ("wrench 25mm", "25mm"),
    ("6m cable", "6m"),
    ("screwdriver 3mm", "3mm"),
    ("bolt 12cm", "12cm"),
    ("rope 2 ft", "2 ft"),
    ("pipe 1.5m", "1.5m"),
    ("bracket 40mm", "40mm"),
    ("rod 7 in", "7 in"),
    ("washer 0.5mm", "0.5mm"),
    ("chain 10m", "10m"),
    ("panel 4 cm", "4 cm"),
    ("pin 18mm", "18mm"),
]

benchmark_row_counts = [3, 6, 12]
benchmark_configs = [
    {
        "name": "serial temp 0.2",
        "threads": 1,
        "temperature": 0.2,
        "seed": 1,
        "retries": 2,
    },
    {
        "name": "parallel temp 0.2",
        "threads": 3,
        "temperature": 0.2,
        "seed": 1,
        "retries": 2,
    },
    {
        "name": "parallel temp 0",
        "threads": 3,
        "temperature": 0,
        "seed": 1,
        "retries": 2,
    },
]

In [ ]:
def run_extract_benchmark(approach, rows, expected, config):
    kwargs = {
        "model": model,
        "seed": config.get("seed"),
        "temperature": config.get("temperature"),
        "timeout": 60,
        "retries": config.get("retries", 2),
        "threads": config.get("threads", 1),
    }
    if approach == "Chat Completions":
        kwargs["url"] = "https://api.openai.com/v1/chat/completions"

    start = time.perf_counter()
    results = extract.ai(
        rows,
        api_key=api_key,
        output=output_schema,
        **kwargs,
    )
    elapsed = time.perf_counter() - start

    values = [row.get("length", "") for row in results]
    correct = [value == expected_value for value, expected_value in zip(values, expected)]
    non_empty = [bool(str(value).strip()) for value in values]
    error_like = [
        str(value).startswith(("Invalid structured response", "Unsupported", "Timed Out", "Failed"))
        for value in values
    ]

    return {
        "approach": approach,
        "config": config["name"],
        "rows": len(rows),
        "accuracy": sum(correct) / len(rows),
        "correct rows": sum(correct),
        "non-empty rate": sum(non_empty) / len(rows),
        "error-like rate": sum(error_like) / len(rows),
        "elapsed seconds": round(elapsed, 3),
        "rows per second": round(len(rows) / elapsed, 3),
        "seconds per row": round(elapsed / len(rows), 3),
        "results": values,
    }

In [ ]:
benchmark_rows = []
for row_count in benchmark_row_counts:
    rows = [case[0] for case in benchmark_cases[:row_count]]
    expected = [case[1] for case in benchmark_cases[:row_count]]
    for config in benchmark_configs:
        benchmark_rows.append(
            run_extract_benchmark("Responses API", rows, expected, config)
        )
        benchmark_rows.append(
            run_extract_benchmark("Chat Completions", rows, expected, config)
        )

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.drop(columns=["results"])

In [ ]:
benchmark_pivot = benchmark_df.pivot_table(
    index=["rows", "config"],
    columns="approach",
    values=["accuracy", "non-empty rate", "error-like rate", "seconds per row"],
)
benchmark_pivot

Use the summary table to look for practical wins: equal or better accuracy, fewer error-like outputs, and lower seconds per row. The detailed `results` column remains in `benchmark_df` if you want to inspect exact row-level differences.

## Benchmark charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharex=True)

for approach, group in benchmark_df.groupby("approach"):
    summary = group.groupby("rows", as_index=False).agg(
        accuracy=("accuracy", "mean"),
        seconds_per_row=("seconds per row", "mean"),
        error_like_rate=("error-like rate", "mean"),
    )
    axes[0].plot(summary["rows"], summary["accuracy"], marker="o", label=approach)
    axes[1].plot(summary["rows"], summary["seconds_per_row"], marker="o", label=approach)
    axes[2].plot(summary["rows"], summary["error_like_rate"], marker="o", label=approach)

axes[0].set_title("Accuracy by row count")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[1].set_title("Latency by row count")
axes[1].set_ylabel("Seconds per row")
axes[2].set_title("Error-like outputs")
axes[2].set_ylabel("Rate")
axes[2].set_ylim(0, 1)

for axis in axes:
    axis.set_xlabel("Rows")
    axis.legend()
    axis.grid(True, alpha=0.25)

plt.tight_layout()

In [ ]:
chart_df = benchmark_df.copy()
chart_df["scenario"] = chart_df["rows"].astype(str) + " rows | " + chart_df["config"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for approach, group in chart_df.groupby("approach"):
    ordered = group.sort_values(["rows", "config"])
    axes[0].plot(ordered["scenario"], ordered["accuracy"], marker="o", label=approach)
    axes[1].plot(ordered["scenario"], ordered["seconds per row"], marker="o", label=approach)

axes[0].set_title("Accuracy by benchmark scenario")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[1].set_title("Latency by benchmark scenario")
axes[1].set_ylabel("Seconds per row")

for axis in axes:
    axis.legend()
    axis.grid(True, alpha=0.25)

plt.xticks(rotation=35, ha="right")
plt.tight_layout()

## Implemented best practices comparison

This table compares the old and new behavior for each OpenAI best practice implemented in this branch.

In [ ]:
best_practices_df = pd.DataFrame(
    [
        {
            "best practice": "Responses API by default",
            "old Chat Completions path": "Only available when callers used the Chat Completions endpoint directly.",
            "new Responses path": "Default OpenAI endpoint is /v1/responses.",
            "why it matters": "Aligns extract.ai with OpenAI's current API surface while keeping the legacy override available.",
        },
        {
            "best practice": "Structured Outputs",
            "old Chat Completions path": "Used a function tool schema through tools/function calling.",
            "new Responses path": "Uses text.format with a strict JSON Schema.",
            "why it matters": "Makes the requested output contract explicit in the Responses API format.",
        },
        {
            "best practice": "Schema sanitization",
            "old Chat Completions path": "Passed the library schema to the function tool path.",
            "new Responses path": "Removes unsupported JSON Schema keys before submitting to OpenAI.",
            "why it matters": "Avoids OpenAI schema errors while preserving user guidance elsewhere when needed.",
        },
        {
            "best practice": "Examples preserved as prompt guidance",
            "old Chat Completions path": "Examples could live in the legacy schema/tool definition.",
            "new Responses path": "Examples are removed from the strict schema and added to instructions as style guidance.",
            "why it matters": "Keeps useful extraction hints without sending unsupported schema keywords.",
        },
        {
            "best practice": "Strict mode by default",
            "old Chat Completions path": "Function schema used strict mode in the legacy path.",
            "new Responses path": "Structured Outputs strict mode is enabled by default.",
            "why it matters": "Reduces missing fields and unexpected extra properties.",
        },
        {
            "best practice": "Dynamic Pydantic validation",
            "old Chat Completions path": "No new local Pydantic validation layer.",
            "new Responses path": "Builds a Pydantic model from the sanitized schema and validates parsed output locally.",
            "why it matters": "Catches bad structured data from mocks, compatible providers, or unexpected model responses.",
        },
        {
            "best practice": "Reasoning controls",
            "old Chat Completions path": "No Responses reasoning parameter.",
            "new Responses path": "Sends low reasoning only for models that support reasoning.",
            "why it matters": "Uses efficient reasoning defaults without breaking models such as gpt-4o-mini.",
        },
        {
            "best practice": "Verbosity controls",
            "old Chat Completions path": "No Responses text.verbosity parameter.",
            "new Responses path": "Sends low verbosity only for models that support low verbosity.",
            "why it matters": "Keeps structured extraction concise without sending unsupported model parameters.",
        },
        {
            "best practice": "Prompt cache key",
            "old Chat Completions path": "No prompt_cache_key.",
            "new Responses path": "Generates a stable prompt_cache_key from namespace, model, and schema.",
            "why it matters": "Makes repeated structured workloads more cache-friendly.",
        },
        {
            "best practice": "Static prompt, dynamic data",
            "old Chat Completions path": "System/user messages and data are sent through the legacy chat shape.",
            "new Responses path": "Instructions and schema stay stable; row data is sent as per-row input.",
            "why it matters": "Improves clarity of what is stable request context versus row-specific data.",
        },
        {
            "best practice": "Unsupported parameter filtering",
            "old Chat Completions path": "Legacy settings preserve Chat Completions-style parameters.",
            "new Responses path": "Filters unsupported Responses params such as seed.",
            "why it matters": "Keeps old recipes working while avoiding Responses API request errors.",
        },
        {
            "best practice": "Backward-compatible legacy override",
            "old Chat Completions path": "Still available by passing a /chat/completions URL.",
            "new Responses path": "Default behavior changes, but the old path remains opt-in.",
            "why it matters": "Reduces migration risk for OpenAI-compatible backends and existing recipes.",
        },
    ]
)

best_practices_df

## Behavior comparison

In [ ]:
pd.DataFrame(
    [
        {
            "area": "Endpoint",
            "new Responses path": "Default OpenAI endpoint",
            "old Chat Completions path": "Only used when caller passes the legacy URL",
        },
        {
            "area": "Output control",
            "new Responses path": "Structured Outputs via text.format JSON Schema",
            "old Chat Completions path": "Function tool schema",
        },
        {
            "area": "Strict schema",
            "new Responses path": "Strict by default, sanitized for OpenAI schema rules",
            "old Chat Completions path": "Legacy strict function schema",
        },
        {
            "area": "Local validation",
            "new Responses path": "Dynamic Pydantic model validates parsed output",
            "old Chat Completions path": "No new local Pydantic layer added",
        },
        {
            "area": "Prompt shape",
            "new Responses path": "Instructions plus input data sent to Responses",
            "old Chat Completions path": "System/user messages plus tool call",
        },
    ]
)

## Local validation failure demo

The new path builds a Pydantic model dynamically from the requested output schema. This cell does not call OpenAI; it demonstrates the same local validation guard that runs after a Responses API result is parsed.

In [ ]:
integer_response_schema = extract._openai_responses.sanitize_schema(
    {
        "type": "object",
        "properties": {
            "count": {
                "type": "integer",
                "description": "Number of matching items",
            }
        },
    }
)

try:
    extract._openai_responses.validate_structured_output(
        {"count": "not a number"},
        integer_response_schema,
    )
except Exception as exc:
    validation_error = str(exc)

validation_error

## What success looks like

The exact values can vary because these are live model calls, but the Responses API column should extract the same length values as the Chat Completions compatibility path for this simple dataset.